In [2]:
import fsspec
import datasets
import pyarrow.parquet as pq

d:\ALL Programming\Hacker_House\EchoQuery-RAG-based-STT\.claude\worktrees\utkarsh-rag-tasklist\03-ai-rag-engine\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
path = "hf://datasets/ai4bharat/MSMARCO-XI@bf5cdc1f26e581e519018e434db14edd1b77602b/train/hintrain.parquet"

parquet_file = pq.ParquetFile(path)

print(parquet_file.schema)

required group field_id=-1 schema {
  optional binary field_id=-1 source_lang (String);
  optional binary field_id=-1 target_lang (String);
  optional group field_id=-1 meta {
    optional int64 field_id=-1 frequency_penalty;
    optional int64 field_id=-1 max_tokens;
    optional binary field_id=-1 model_name (String);
    optional int64 field_id=-1 presence_penalty;
    optional int64 field_id=-1 temperature;
    optional int64 field_id=-1 top_p;
  }
  optional binary field_id=-1 Answer (String);
  optional int64 field_id=-1 query_id;
  optional binary field_id=-1 query_type (String);
  optional group field_id=-1 passages {
    optional group field_id=-1 English_passages (List) {
      repeated group field_id=-1 list {
        optional binary field_id=-1 element (String);
      }
    }
    optional group field_id=-1 Translated_passages (List) {
      repeated group field_id=-1 list {
        optional binary field_id=-1 element (String);
      }
    }
    optional group field_id=-1 is

In [4]:
path = "hf://datasets/ai4bharat/MSMARCO-XI@bf5cdc1f26e581e519018e434db14edd1b77602b/train/hintrain.parquet"

parquet_file = pq.ParquetFile(path)

print("Rows:", parquet_file.metadata.num_rows)
print("Row groups:", parquet_file.num_row_groups)

print("\nRow group metadata:")
print(parquet_file.metadata.row_group(0))

Rows: 778638
Row groups: 1

Row group metadata:
  num_columns: 17
  num_rows: 778638
  total_byte_size: 9729210887
  sorting_columns: ()


In [5]:
for i in range(parquet_file.metadata.num_columns):
    column = parquet_file.metadata.schema.column(i)

    print(
        f"{i:2} | "
        f"{column.name:25} | "
        f"{column.physical_type:10} | "
        f"{column.logical_type}"
    )

 0 | source_lang               | BYTE_ARRAY | String
 1 | target_lang               | BYTE_ARRAY | String
 2 | frequency_penalty         | INT64      | None
 3 | max_tokens                | INT64      | None
 4 | model_name                | BYTE_ARRAY | String
 5 | presence_penalty          | INT64      | None
 6 | temperature               | INT64      | None
 7 | top_p                     | INT64      | None
 8 | Answer                    | BYTE_ARRAY | String
 9 | query_id                  | INT64      | None
10 | query_type                | BYTE_ARRAY | String
11 | element                   | BYTE_ARRAY | String
12 | element                   | BYTE_ARRAY | String
13 | element                   | INT64      | None
14 | Eng_Query                 | BYTE_ARRAY | String
15 | Eng_Answer                | BYTE_ARRAY | String
16 | query                     | BYTE_ARRAY | String


In [6]:
for i in range(parquet_file.metadata.num_columns):
    column = parquet_file.metadata.row_group(0).column(i)

    print(
        f"{i:2} | "
        f"{column.path_in_schema:35} | "
        f"{column.total_compressed_size / (1024**2):10.2f} MB | "
        f"{column.total_uncompressed_size / (1024**2):10.2f} MB"
    )
    

 0 | source_lang                         |       0.00 MB |       0.00 MB
 1 | target_lang                         |       0.00 MB |       0.00 MB
 2 | meta.frequency_penalty              |       0.00 MB |       0.00 MB
 3 | meta.max_tokens                     |       0.00 MB |       0.00 MB
 4 | meta.model_name                     |       0.00 MB |       0.00 MB
 5 | meta.presence_penalty               |       0.00 MB |       0.00 MB
 6 | meta.temperature                    |       0.00 MB |       0.00 MB
 7 | meta.top_p                          |       0.00 MB |       0.00 MB
 8 | Answer                              |      43.11 MB |     137.64 MB
 9 | query_id                            |       3.88 MB |       6.21 MB
10 | query_type                          |       0.28 MB |       0.28 MB
11 | passages.English_passages.list.element |    1324.50 MB |    2529.64 MB
12 | passages.Translated_passages.list.element |    2095.74 MB |    6443.88 MB
13 | passages.is_selected.list.element   |

In [7]:
light_columns = [
    "source_lang",
    "target_lang",
    "query_id",
    "query_type",
    "query",
    "Eng_Query",
    "Answer",
    "Eng_Answer",
]

for batch in parquet_file.iter_batches(
    batch_size=5,
    columns=light_columns
):
    rows = batch.to_pylist()

    for i, row in enumerate(rows, 1):
        print(f"\n{'=' * 60}")
        print(f"RECORD {i}")
        print(f"{'=' * 60}")

        for key, value in row.items():
            print(f"\n--- {key} ---")
            print(value)

    break


RECORD 1

--- source_lang ---
eng_Latn

--- target_lang ---
hin_Deva

--- query_id ---
1185869

--- query_type ---
DESCRIPTION

--- query ---
मैनहट्टन परियोजना की सफलता का तुरंत क्या प्रभाव पड़ा?

--- Eng_Query ---
)what was the immediate impact of the success of the manhattan project?

--- Answer ---
मैनहट्टन परियोजना की सफलता का तत्काल प्रभाव परमाणु शोधकर्ताओं और इंजीनियरों की प्रभावशाली उपलब्धि पर एकमात्र बादल था जो उनकी सफलता का वास्तव में अर्थ था; लाखों निर्दोष जीवन नष्ट हो गए।

--- Eng_Answer ---
The immediate impact of the success of the manhattan project was the only cloud hanging over the impressive achievement of the atomic researchers and engineers is what their success truly meant; hundreds of thousands of innocent lives obliterated.

RECORD 2

--- source_lang ---
eng_Latn

--- target_lang ---
hin_Deva

--- query_id ---
1185868

--- query_type ---
DESCRIPTION

--- query ---
न्याय को पीड़ित, समुदाय और अपराधी द्वारा अपराधी कृत्य के कारण हुए नुकसान की मरम्मत करने के लिए डिज़ा

In [ ]:
passage_columns = [
    "passages.English_passages",
    "passages.Translated_passages",
    "passages.is_selected",
]

columns = [
    "source_lang",
    "target_lang",
    "query_id",
    "query_type",
    "query",
    "Eng_Query",
    "Answer",
    "Eng_Answer",
    *passage_columns,
]

for batch in parquet_file.iter_batches(
    batch_size=5,
    columns=columns
):
    rows = batch.to_pylist()

    for i, row in enumerate(rows, 1):
        print(f"\n{'=' * 80}")
        print(f"RECORD {i}")
        print(f"{'=' * 80}")

        for key, value in row.items():
            print(f"\n--- {key} ---")
            print(value)

    break

In [8]:
hindi_path = path

pf = pq.ParquetFile(hindi_path)

table = pf.read(
    columns=["source_lang", "target_lang"],
)

df = table.to_pandas()

print(df["source_lang"].value_counts())
print()
print(df["target_lang"].value_counts())

source_lang
eng_Latn    778638
Name: count, dtype: int64

target_lang
hin_Deva    778638
Name: count, dtype: int64


In [9]:
print(hindi_path)

hf://datasets/ai4bharat/MSMARCO-XI@bf5cdc1f26e581e519018e434db14edd1b77602b/train/hintrain.parquet


In [10]:
hf_path = path
with fsspec.open(hf_path, "rb") as f:
    pf = pq.ParquetFile(f)

    batch = next(
        pf.iter_batches(
            batch_size=5,
            columns=[
                "source_lang",
                "target_lang",
                "query_id",
                "query_type",
                "query",
                "Eng_Query",
                "Answer",
                "Eng_Answer",
            ],
        )
    )

for i, row in enumerate(batch.to_pylist(), 1):
    print(f"\n{'=' * 30}")
    print(f"RECORD {i}")
    print(f"{'=' * 30}")

    for key, value in row.items():
        print(f"{key}: {value}")


RECORD 1
source_lang: eng_Latn
target_lang: hin_Deva
query_id: 1185869
query_type: DESCRIPTION
query: मैनहट्टन परियोजना की सफलता का तुरंत क्या प्रभाव पड़ा?
Eng_Query: )what was the immediate impact of the success of the manhattan project?
Answer: मैनहट्टन परियोजना की सफलता का तत्काल प्रभाव परमाणु शोधकर्ताओं और इंजीनियरों की प्रभावशाली उपलब्धि पर एकमात्र बादल था जो उनकी सफलता का वास्तव में अर्थ था; लाखों निर्दोष जीवन नष्ट हो गए।
Eng_Answer: The immediate impact of the success of the manhattan project was the only cloud hanging over the impressive achievement of the atomic researchers and engineers is what their success truly meant; hundreds of thousands of innocent lives obliterated.

RECORD 2
source_lang: eng_Latn
target_lang: hin_Deva
query_id: 1185868
query_type: DESCRIPTION
query: न्याय को पीड़ित, समुदाय और अपराधी द्वारा अपराधी कृत्य के कारण हुए नुकसान की मरम्मत करने के लिए डिज़ाइन किया गया है। प्रश्न 19 विकल्प:
Eng_Query: _________ justice is designed to repair the harm to victim,

In [ ]:
import pyarrow.parquet as pq
import fsspec

hf_path = hindi_path

with fsspec.open(hf_path, "rb") as f:
    pf = pq.ParquetFile(hf_path if False else f)

    batch = next(
        pf.iter_batches(
            batch_size=5,
            columns=["query_id", "query", "Eng_Query", "Answer", "Eng_Answer", "passages"],
        )
    )

rows = batch.to_pylist()

for i, row in enumerate(rows, 1):
    passages = row["passages"]

    print(f"\n{'=' * 80}")
    print(f"RECORD {i} | query_id={row['query_id']}")
    print(f"{'=' * 80}")

    print(f"Query: {row['query']}")
    print(f"English Query: {row['Eng_Query']}")

    print("\nPassage counts:")
    print("English:", len(passages["English_passages"]))
    print("Translated:", len(passages["Translated_passages"]))
    print("Labels:", len(passages["is_selected"]))

    print("\nSelection labels:")
    print(passages["is_selected"])

    print("\nThis is English passage:")
    print(passages["English_passages"][0])

    print("\nThis is Hindi passage:")
    print(passages["Translated_passages"][0])


RECORD 1 | query_id=1185869
Query: मैनहट्टन परियोजना की सफलता का तुरंत क्या प्रभाव पड़ा?
English Query: )what was the immediate impact of the success of the manhattan project?

Passage counts:
English: 10
Translated: 10
Labels: 10

Selection labels:
[1, 0, 0, 0, 0, 0, 0, 0, 0, 0]

This is English passage:
The presence of communication amid scientific minds was equally important to the success of the Manhattan Project as scientific intellect was. The only cloud hanging over the impressive achievement of the atomic researchers and engineers is what their success truly meant; hundreds of thousands of innocent lives obliterated.

This is Hindi passage:
वैज्ञानिक दिमाग के बीच संचार की उपस्थिति मैनहट्टन परियोजना की सफलता के लिए उतनी ही महत्वपूर्ण थी जितनी कि वैज्ञानिक बुद्धिमत्ता थी। परमाणु शोधकर्ताओं और इंजीनियरों की प्रभावशाली उपलब्धि पर लटकता एकमात्र बादल उनकी सफलता का वास्तव में क्या अर्थ था; सैकड़ों हजारों निर्दोष जीवन का विनाश।

RECORD 2 | query_id=1185868
Query: न्याय को पीड़ित, समुद

In [11]:
import pyarrow.parquet as pq
import fsspec
from collections import Counter

hf_path = "hf://datasets/ai4bharat/MSMARCO-XI@bf5cdc1f26e581e519018e434db14edd1b77602b/train/hintrain.parquet"

with fsspec.open(hf_path, "rb") as f:
    pf = pq.ParquetFile(f)

    selected_counts = Counter()
    total_passages = 0
    total_selected = 0

    batch_size = 1000

    for batch in pf.iter_batches(
        batch_size=batch_size,
        columns=["query_id", "passages"],
    ):
        for row in batch.to_pylist():
            labels = row["passages"]["is_selected"]

            selected = sum(labels)

            selected_counts[selected] += 1
            total_passages += len(labels)
            total_selected += selected

print("Queries analysed:", sum(selected_counts.values()))
print("Total passages:", total_passages)
print("Total selected passages:", total_selected)

print("\nSelected passages per query:")
for count in sorted(selected_counts):
    print(f"{count} selected: {selected_counts[count]} queries")

Queries analysed: 778638
Total passages: 7769498
Total selected passages: 513004

Selected passages per query:
0 selected: 294369 queries
1 selected: 459837 queries
2 selected: 21071 queries
3 selected: 2613 queries
4 selected: 592 queries
5 selected: 126 queries
6 selected: 22 queries
7 selected: 8 queries


In [12]:
import pyarrow.parquet as pq
import fsspec

hf_path = (
    "hf://datasets/ai4bharat/MSMARCO-XI@"
    "bf5cdc1f26e581e519018e434db14edd1b77602b"
    "/train/hintrain.parquet"
)

zero_selected = 0
zero_with_answer = 0
zero_no_answer = 0

examples_with_answer = []
examples_no_answer = []

with fsspec.open(hf_path, "rb") as f:
    pf = pq.ParquetFile(f)

    for batch in pf.iter_batches(
        batch_size=1000,
        columns=["query_id", "query", "Answer", "passages"],
    ):
        for row in batch.to_pylist():

            labels = row["passages"]["is_selected"]

            if sum(labels) == 0:
                zero_selected += 1

                answer = (row["Answer"] or "").strip()

                if answer.lower() == "no answer present.":
                    zero_no_answer += 1

                    if len(examples_no_answer) < 5:
                        examples_no_answer.append(row)
                else:
                    zero_with_answer += 1

                    if len(examples_with_answer) < 5:
                        examples_with_answer.append(row)

print("Zero-selected queries:", zero_selected)
print("Zero-selected + 'No Answer Present':", zero_no_answer)
print("Zero-selected + actual answer:", zero_with_answer)

print("\nExamples: zero-selected + actual answer")
for row in examples_with_answer:
    print("\nquery_id:", row["query_id"])
    print("query:", row["query"])
    print("answer:", row["Answer"])

print("\nExamples: zero-selected + No Answer Present")
for row in examples_no_answer:
    print("\nquery_id:", row["query_id"])
    print("query:", row["query"])
    print("answer:", row["Answer"])

Zero-selected queries: 294369
Zero-selected + 'No Answer Present': 0
Zero-selected + actual answer: 294369

Examples: zero-selected + actual answer

query_id: 620830
query: फ्लूम किस दिशा में बहता है
answer: कोई उत्तर नहीं मिला।

query_id: 623214
query: स्नातक छात्र कक्षा में क्या पहनते हैं
answer: कोई उत्तर नहीं मिला।

query_id: 1164716
query: क्या आप विस्तृत कटौती प्रपत्र पर राज्य के लिए भुगतान किए गए सी.पी.ए. शुल्क का दावा कर सकते हैं?
answer: कोई उत्तर नहीं मिला।

query_id: 385672
query: सिसडेट का उपयोग कैसे करें?
answer: कोई उत्तर नहीं मिला।

query_id: 322283
query: डरहम, ओंटारियो में एक सुरक्षा अधिकारी को किराए पर लेने के लिए कितना होगा?
answer: कोई उत्तर नहीं मिला।

Examples: zero-selected + No Answer Present


In [13]:
print(hindi_path)

hf://datasets/ai4bharat/MSMARCO-XI@bf5cdc1f26e581e519018e434db14edd1b77602b/train/hintrain.parquet


In [14]:
from collections import Counter

hf_path = (
    "hf://datasets/ai4bharat/MSMARCO-XI@"
    "bf5cdc1f26e581e519018e434db14edd1b77602b"
    "/train/hintrain.parquet"
)

zero_selected_answers = Counter()

with fsspec.open(hf_path, "rb") as f:
    pf = pq.ParquetFile(f)

    for batch in pf.iter_batches(
        batch_size=1000,
        columns=["Answer", "passages"],
    ):
        for row in batch.to_pylist():

            labels = row["passages"]["is_selected"]

            if sum(labels) == 0:
                answer = (row["Answer"] or "").strip()
                zero_selected_answers[answer] += 1

print("Number of distinct answers among zero-selected queries:")
print(len(zero_selected_answers))

print("\nAnswers:")
for answer, count in zero_selected_answers.most_common(20):
    print(f"{count:>8} | {repr(answer)}")

Number of distinct answers among zero-selected queries:
47

Answers:
  293952 | 'कोई उत्तर नहीं मिला।'
      68 | 'क्या आप किसी के काम के लिए किसी के काम को करने के लिए किसी के काम को करने के लिए किसी के काम को करने के लिए किसी के काम को करने के लिए किसी के काम को करने के लिए किसी के काम को करने के लिए किसी के काम को करने के लिए किसी के काम को करने के लिए किसी के काम को करने के लिए किसी के काम को करने के लिए किसी के काम को करने के लिए किसी के काम को करने के लिए किसी के काम को करने के लिए किसी के काम को करने के लिए किसी के काम को करने के लिए किसी के काम को करने के लिए किसी के काम को करने के लिए किसी के काम को करने के लिए किसी के काम को करने के लिए किसी के काम को करने के लिए किसी के काम को करने के लिए किसी के काम को करने के लिए किसी के काम को करने के लिए किसी के काम को करने के लिए किसी के काम को करने के लिए किसी के काम को करने के लिए किसी के काम को करने के लिए किसी के काम को करने के लिए किसी के काम को करने के लिए किसी के काम को करने के लिए किसी के काम को करने के लिए किसी के काम को करने क

In [16]:
from datasets import load_dataset

hindi_train = load_dataset(
    "ai4bharat/MSMARCO-XI",
    split="train",
    streaming=True,
    trust_remote_code=False,
)

Repo card metadata block was not found. Setting CardData to empty.


In [17]:
zero_selected_examples = []

for record in hindi_train:
    labels = record["passages"]["is_selected"]

    if sum(labels) == 0:
        zero_selected_examples.append({
            "query_id": record["query_id"],
            "query": record["query"],
            "eng_query": record["Eng_Query"],
            "answer": record["Answer"],
            "eng_answer": record["Eng_Answer"],
            "english_passages": record["passages"]["English_passages"],
            "translated_passages": record["passages"]["Translated_passages"],
            "labels": labels,
        })

        if len(zero_selected_examples) >= 10:
            break

len(zero_selected_examples)

ArrowNotImplementedError: Nested data conversions not implemented for chunked array outputs

In [19]:
import pyarrow.parquet as pq
from collections import Counter

pf = pq.ParquetFile(hindi_path)

stats = Counter()

examples = {
    "zero_selected_no_answer": [],
    "zero_selected_actual_answer": []
}

for batch in pf.iter_batches(
    batch_size=1000,
    columns=["query_id", "query", "Answer", "Eng_Answer", "passages"]
):
    for row in batch.to_pylist():

        labels = row["passages"]["is_selected"]

        if sum(labels) != 0:
            continue

        stats["zero_selected"] += 1

        answer = (row["Answer"] or "").strip()
        eng_answer = (row["Eng_Answer"] or "").strip()

        # Dataset's explicit no-answer marker
        if answer == "कोई उत्तर नहीं मिला।" or eng_answer == "No Answer Present.":
            stats["zero_selected_no_answer"] += 1

            if len(examples["zero_selected_no_answer"]) < 5:
                examples["zero_selected_no_answer"].append({
                    "query_id": row["query_id"],
                    "query": row["query"],
                    "answer": answer,
                    "eng_answer": eng_answer
                })

        else:
            stats["zero_selected_actual_answer"] += 1

            if len(examples["zero_selected_actual_answer"]) < 5:
                examples["zero_selected_actual_answer"].append({
                    "query_id": row["query_id"],
                    "query": row["query"],
                    "answer": answer,
                    "eng_answer": eng_answer
                })


print("=" * 70)
print("2A — ZERO-SELECTED ANSWERABILITY ANALYSIS")
print("=" * 70)

print(f"\nZero-selected queries:           {stats['zero_selected']}")
print(f"Zero-selected + No Answer:       {stats['zero_selected_no_answer']}")
print(f"Zero-selected + Actual Answer:   {stats['zero_selected_actual_answer']}")

print("\n--- Zero-selected + No Answer examples ---")
for x in examples["zero_selected_no_answer"]:
    print(f"\nquery_id: {x['query_id']}")
    print(f"query:    {x['query']}")
    print(f"answer:   {x['answer']}")

print("\n--- Zero-selected + Actual Answer examples ---")
for x in examples["zero_selected_actual_answer"]:
    print(f"\nquery_id: {x['query_id']}")
    print(f"query:    {x['query']}")
    print(f"answer:   {x['answer']}")
    

2A — ZERO-SELECTED ANSWERABILITY ANALYSIS

Zero-selected queries:           294369
Zero-selected + No Answer:       293952
Zero-selected + Actual Answer:   417

--- Zero-selected + No Answer examples ---

query_id: 620830
query:    फ्लूम किस दिशा में बहता है
answer:   कोई उत्तर नहीं मिला।

query_id: 623214
query:    स्नातक छात्र कक्षा में क्या पहनते हैं
answer:   कोई उत्तर नहीं मिला।

query_id: 1164716
query:    क्या आप विस्तृत कटौती प्रपत्र पर राज्य के लिए भुगतान किए गए सी.पी.ए. शुल्क का दावा कर सकते हैं?
answer:   कोई उत्तर नहीं मिला।

query_id: 385672
query:    सिसडेट का उपयोग कैसे करें?
answer:   कोई उत्तर नहीं मिला।

query_id: 322283
query:    डरहम, ओंटारियो में एक सुरक्षा अधिकारी को किराए पर लेने के लिए कितना होगा?
answer:   कोई उत्तर नहीं मिला।

--- Zero-selected + Actual Answer examples ---

query_id: 624861
query:    घोंघे पर्यावरण के लिए क्या करते हैं
answer:   क्या आप किसी के काम के लिए कोई काम करने के लिए किसी के काम के लिए कोई काम करने के लिए किसी के काम के लिए कोई काम करन

In [20]:
import pyarrow.parquet as pq

zero_selected_examples = []

pf = pq.ParquetFile(hindi_path)

for batch in pf.iter_batches(
    batch_size=1000,
    columns=[
        "query_id",
        "query",
        "Eng_Query",
        "Answer",
        "Eng_Answer",
        "passages",
    ],
):
    for record in batch.to_pylist():

        labels = record["passages"]["is_selected"]

        if sum(labels) == 0:

            zero_selected_examples.append({
                "query_id": record["query_id"],
                "query": record["query"],
                "eng_query": record["Eng_Query"],
                "answer": record["Answer"],
                "eng_answer": record["Eng_Answer"],
                "english_passages": record["passages"]["English_passages"],
                "translated_passages": record["passages"]["Translated_passages"],
                "labels": labels,
            })

            # We only need 10 examples
            if len(zero_selected_examples) >= 10:
                break

    if len(zero_selected_examples) >= 10:
        break

print("Collected examples:", len(zero_selected_examples))

Collected examples: 10


In [21]:
for i, record in enumerate(zero_selected_examples, 1):

    print("=" * 80)
    print(f"ZERO-SELECTED RECORD {i}")
    print("=" * 80)

    print(f"query_id:   {record['query_id']}")
    print(f"query:      {record['query']}")
    print(f"Eng_Query:  {record['eng_query']}")
    print(f"Answer:     {record['answer']}")
    print(f"Eng_Answer: {record['eng_answer']}")
    print(f"Labels:     {record['labels']}")

    print("\nPASSAGES:")
    for j, (eng, hin, label) in enumerate(
        zip(
            record["english_passages"],
            record["translated_passages"],
            record["labels"]
        ),
        1
    ):
        print(f"\n--- Passage {j} | selected={label} ---")
        print("EN:", eng)
        print("HI:", hin)

    print()

ZERO-SELECTED RECORD 1
query_id:   620830
query:      फ्लूम किस दिशा में बहता है
Eng_Query:  what direction does phloem flow
Answer:     कोई उत्तर नहीं मिला।
Eng_Answer: No Answer Present.
Labels:     [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

PASSAGES:

--- Passage 1 | selected=0 ---
EN: Phloem is a conductive (or vascular) tissue found in plants. Phloem carries the products of photosynthesis (sucrose and glucose) from the leaves to other parts of the plant. … The corresponding system that circulates water and minerals from the roots is called the xylem.
HI: फ्लोएम पौधों में पाया जाने वाला एक संवाहक (या संवहनी) ऊतक है। फ्लोएम पत्तियों से पौधे के अन्य भागों तक प्रकाश संश्लेषण (सुक्रोज और ग्लूकोज) के उत्पादों को ले जाता है। जड़ों से पानी और खनिजों को प्रसारित करने वाली संबंधित प्रणाली को जाइलम कहा जाता है।

--- Passage 2 | selected=0 ---
EN: Phloem and xylem are complex tissues that perform transportation of food and water in a plant. They are the vascular tissues of the plant and together form va